# Data Sourcing

<a target="_blank" href="https://colab.research.google.com/github/zentralbibliothek-zuerich/zblab-summerschool-2026/blob/main/notebooks/01_data_sourcing.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


This notebook contains boilerplate code that you don't need to interact with directly.

The sections where you can safely experiment or customize are clearly marked with such comments:

```python
# ⬇️⬇️⬇️
YOUR_INPUT = ""
# ⬆️⬆️⬆️
```

## Setup

### Housekeeping (no interaction required)

In [ ]:
%pip install impresso
%pip install simplemma
%pip install nltk

❗ Please restart the kernel/runtime after installing the package to ensure that all changes take effect.

(Google Colab might initiate a restart on its own)

In [1]:
import os
import random
import time
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

In [2]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBLabSummerSchool2026/data') if IN_COLAB else Path('../data')

In [ ]:
def confirm(question: str = "Do you want to execute this cell?") -> bool:
    """Prompt the user for confirmation before executing a cell."""
    while True:
        response = input(f"{question} (y/n): ").lower()
        if response in ["y", "yes"]:
            return True
        elif response in ["n", "no"]:
            return False
        else:
            print("Please enter 'y' or 'n'.")

<img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=12> The cell below will attempt to connect to your Google Drive.

*Once prompted, give all demanded permissions*

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DATA_DIR, exist_ok=True)

### Setup (interaction required)

#### Connect with the impresso API

You can generate an API Access Token in [Impresso Datalab](https://impresso-project.ch/datalab/token/).

The cell below will prompt you for this token.

In [5]:
import impresso

client = impresso.connect()


Click on the following link to access the login page: https://impresso-project.ch/datalab/token
 - 🔤 Enter your email/password on this page.
 - 🔑 Once logged in, a secret token will be generated for you.
 - 📋 Copy this token and paste it into the input field below. Then press "Enter". 👇🏼.

The provided token is invalid. Have you entered it correctly? 🤔


ValueError: The provided token is invalid. Have you entered it correctly? 🤔

## Explore impresso

Play around in the [Impresso App](https://impresso-project.ch/app) and try to find a query that creates a small and interesting corpus of around 4000 articles.

Depending on your interest, try to create a query that returns data that is temporally dispersed.

![temporal-dispersion](../assets/impresso_temporal_dispersion.png)

This can mean that you need to combine different spellings or old terms in your query, for example adding *Armuth* to an *Armut*-query.

Once you have found an interesting query, you can click "Try in Datalab" to extract the search terms and enter them below.

![try-in-datalab](../assets/)


## Create your corpus

In [ ]:
# ⬇️⬇️⬇️ Your search query here
QUERY_PARAMS = {
    "with_text_contents": True,
    "language": "de",
    "term": impresso.OR(["Armenpflege"]),
}
# ⬆️⬆️⬆️

In [7]:
# ⬇️⬇️⬇️  This name will be used to save your files and throughout the rest of the summer school.
CORPUS_NAME = "armenpflege"
# ⬆️⬆️⬆️ 

### Create an index

First, we create an index of all articles defined by your query.

The index will not contain the fulltext of the articles, but it will contain the *"id"* of each article.
This complete list of ids will be used in the next step to download the fulltext.

In [ ]:
search_results = client.search.find(
	**QUERY_PARAMS,
)

if search_results.total > 5000:
    print(f"⚠️ Warning: Your search query returned {search_results.total} results!")
    print("The download will take a long time, and not all modules of the summer school may be feasible to execute.")
    
search_results

In [9]:
INDEXPATH = DATA_DIR / f"{CORPUS_NAME}.index.csv"

In [ ]:
TOTAL_RESULTS = search_results.total
LIMIT = 250 # Maximum is 1000

all_results = []
n_retrieved = 0

for offset in range(0, TOTAL_RESULTS, LIMIT):
    print(f"Retrieved {n_retrieved} / {TOTAL_RESULTS} items so far...", end="\r")

    search_results = client.search.find(
        **QUERY_PARAMS,
        limit=LIMIT,
        offset=offset
    )

    all_results.append(search_results.df)
    n_retrieved += len(search_results.df)

    print(f"Retrieved {n_retrieved} / {TOTAL_RESULTS} items so far...", end="\r")

index_df: pd.DataFrame = pd.concat(all_results)

# Save only unique index values, for resuming after interruptions.
index_df = index_df[~index_df.index.duplicated(keep='first')]
index_df.to_csv(INDEXPATH, index=True)

### Download the full text for all found articles

The cell below determines how many articles still need to be retrieved.

That way, if the download is interrupted at any point, we don't need to re-download everything that has already been saved.

In [16]:
TIMEOUT = 2 # Be nice to the server and avoid sending too many requests in a short time
OUTPATH = DATA_DIR / f"{CORPUS_NAME}.wide.parquet"

def load_missing_ids(print_info: bool = True) -> tuple[list[str], pd.DataFrame]:
    try:
        index_df = pd.read_csv(INDEXPATH, index_col="id")
        wanted_ids = set(index_df.index.to_list())
    except FileNotFoundError:
        wanted_ids = set()

    try:
        fulltext_df = pd.read_parquet(OUTPATH)
        retrieved_ids = set(fulltext_df.index.to_list())
    except FileNotFoundError:
        fulltext_df = pd.DataFrame()
        retrieved_ids = set()

    missing_ids = [id for id in wanted_ids if id not in retrieved_ids]

    if print_info:
        print(f"Missing {len(missing_ids)} items. (Already retrieved {len(retrieved_ids)})")
        print(f"Requests for missing items will take approximately {len(missing_ids) * TIMEOUT / 60:.2f} minutes (assuming ~{1 / TIMEOUT:.2f} request per second).")

    return missing_ids, fulltext_df

missing_ids, fulltext_df = load_missing_ids()

Missing 0 items. (Already retrieved 6972)
Requests for missing items will take approximately 0.00 minutes (assuming ~0.50 request per second).


Retrieve all missing documents

In [11]:
# To avoid accidental overwrites, this cell loads the data again in the beginning
missing_ids, fulltext_df = load_missing_ids()

try:
    for i, doc_id in enumerate(missing_ids):
        time.sleep(random.uniform(TIMEOUT * 0.5, TIMEOUT * 1.5))
        print(f"({i+1}/{len(missing_ids)}) - Processing document {doc_id}", end="\r")
        content_item = client.content_items.get(doc_id)
        content_item_df = content_item.df
        content_item_df = content_item_df.set_index("id")
        fulltext_df = pd.concat([fulltext_df, content_item_df], ignore_index=False)
        
        if (i + 1) % 100 == 0: # Save progress every 100 items
            fulltext_df.to_parquet(OUTPATH, index=True)
except KeyboardInterrupt:
    print(f"Process interrupted by user. Saving progress...                              ")
    fulltext_df.to_parquet(OUTPATH, index=True)
except Exception as e:
    print(f"An error occurred. Saving progress...                              ")
    fulltext_df.to_parquet(OUTPATH, index=True)
    raise e
fulltext_df.to_parquet(OUTPATH, index=True)

Missing 0 items. (Already retrieved 6972)
Requests for missing items will take approximately 0.00 minutes (assuming ~0.50 request per second).


### Save the corpus with fewer columns

Saving the corpus with only the needed columns saves loading times and memory down the line.
Furthermore, we rename the columns to a more practical naming scheme.

For the rest of the summer school, we will only be working with the smaller version created below.

In [20]:
reduced_df = fulltext_df[["meta.date", "meta.countryCode", "meta.provinceCode", "meta.mediaTitle", "text.langCode", "text.content"]]

reduced_df = reduced_df.rename(columns={
    "meta.date": "date",
    "meta.countryCode": "country_code",
    "meta.provinceCode": "province_code",
    "meta.mediaTitle": "media_title",
    "text.langCode": "lang_code",
    "text.content": "content",
})

# reduced_df.to_csv(os.path.join(DATA_DIR, f"{CORPUS_NAME}.raw.csv"), index=True)
reduced_df.to_parquet(os.path.join(DATA_DIR, f"{CORPUS_NAME}.full.parquet"), index=True)